In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [12]:
def get_html(url = 'https://sindipetroprsc.org.br/noticias/'):
    payload = {}
    headers = {
        'Cookie': 'last_page_visited=https%3A%2F%2Fsindipetroprsc.org.br%2Fnoticias%2F'
    }

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [13]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%d de %B de %Y")
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue


    return news_links

In [15]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [18]:
def get_next_page(fnp_url, next_page_number = 1):
    validated_news_links = []
    url = fnp_url + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [43]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                            .replace('\xa0',' ')\
                                                            .replace('\n',' ')\
                                                            .replace('\t',' ')\
                                                            .replace('[email-protected]', '')\
                                                            .strip() \
                                                            for paragraph in paragraphs] \
                if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0

    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [ ]:
def remove_empty_pragraphs(paragraphs):
    
    return paragraphs

def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')
    #paragraphs = sanitize_paragraphs(paragraphs)

    return title, paragraphs

In [48]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://sindipetroprsc.org.br/noticias/'
    url = url_default + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'PR_SC',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1
        break

    return result

In [49]:
result = main()
result

  0%|                                                                                                                                                                                                                                                   | 0/72 [00:02<?, ?it/s]


[{'sindicato': 'PR_SC',
  'url': 'https://sindipetroprsc.org.br/meu-brasil-brasileiro-curitiba-vai-as-ruas-no-dia-29-08-em-defesa-da-soberania-nacional/',
  'titulo': '“Meu Brasil Brasileiro”: Curitiba vai às ruas no dia 29/08 em defesa da soberania nacional',
  'data': datetime.datetime(2025, 8, 21, 0, 0),
  'paragrafo': <p>Curitiba terá, no dia 29 de agosto, um novo capítulo da mobilização em defesa da soberania nacional. O protesto “Meu Brasil Brasileiro” convoca a população a se somar em um ato plural na Praça Santos Andrade, a partir das 17h, com a mensagem de que “o Brasil é dos brasileiros e das brasileiras” e que não se aceitará interferência externa nos rumos do país.</p>,
  'num_paragrafo': 1},
 {'sindicato': 'PR_SC',
  'url': 'https://sindipetroprsc.org.br/meu-brasil-brasileiro-curitiba-vai-as-ruas-no-dia-29-08-em-defesa-da-soberania-nacional/',
  'titulo': '“Meu Brasil Brasileiro”: Curitiba vai às ruas no dia 29/08 em defesa da soberania nacional',
  'data': datetime.dateti